In [1]:
# Import Required Libraries
import pandas as pd
import numpy as np

In [2]:
# Load Dataset
df = pd.read_csv("../data/processed/london_listings_cleaned.csv")

df.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,13913,https://www.airbnb.com/rooms/13913,20250914034649,2025-09-16,city scrape,Holiday London DB Room Let-on going,My bright double bedroom with a large window h...,Finsbury Park is a friendly melting pot commun...,https://a0.muscache.com/pictures/miso/Hosting-...,54730,...,4.87,4.78,4.78,NaN,f,2,1,1,0,0.30
1,15400,https://www.airbnb.com/rooms/15400,20250914034649,2025-09-16,city scrape,Bright Chelsea Apartment. Chelsea!,Lots of windows and light. St Luke's Gardens ...,It is Chelsea.,https://a0.muscache.com/pictures/428392/462d26...,60302,...,4.84,4.93,4.74,NaN,f,1,1,0,0,0.51
2,17402,https://www.airbnb.com/rooms/17402,20250914034649,2025-09-16,city scrape,Very Central Modern 3-Bed/2 Bath By Oxford St W1,"You'll have a great time in this beautiful, cl...","Fitzrovia is a very desirable trendy, arty and...",https://a0.muscache.com/pictures/39d5309d-fba7...,67564,...,4.72,4.89,4.61,NaN,f,2,2,0,0,0.32
3,24328,https://www.airbnb.com/rooms/24328,20250914034649,2025-09-18,previous scrape,Battersea live/work artist house,"Artist house by SW Battersea Park, bright high...","- Battersea is a quiet family area, easy acces...",https://a0.muscache.com/pictures/9194b40f-c627...,41759,...,4.93,4.60,4.65,NaN,f,1,1,0,0,0.53
4,36274,https://www.airbnb.com/rooms/36274,20250914034649,2025-09-15,city scrape,Bright 1 bedroom apt off brick lane in Shoreditch,*Update June '25- Pump Installed to improve wa...,NaN,https://a0.muscache.com/pictures/hosting/Hosti...,133271,...,4.46,4.85,4.54,NaN,t,2,2,0,0,0.09


In [3]:
# Dataset Overview
print("Dataset Shape:", df.shape)

df.info()

Dataset Shape: (96871, 79)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 96871 entries, 0 to 96870
Data columns (total 79 columns):
 #   Column                                        Non-Null Count  Dtype  
---  ------                                        --------------  -----  
 0   id                                            96871 non-null  int64  
 1   listing_url                                   96871 non-null  object 
 2   scrape_id                                     96871 non-null  int64  
 3   last_scraped                                  96871 non-null  object 
 4   source                                        96871 non-null  object 
 5   name                                          96871 non-null  object 
 6   description                                   94421 non-null  object 
 7   neighborhood_overview                         41208 non-null  object 
 8   picture_url                                   96865 non-null  object 
 9   host_id                           

In [4]:
# Data Quality Assessment

# Check Missing Values
missing = (
    df.isnull()
      .sum()
      .sort_values(ascending=False)
)

missing[missing > 0]

license                         96871
calendar_updated                96871
neighbourhood_group_cleansed    96871
neighborhood_overview           55663
neighbourhood                   55662
host_neighbourhood              51021
host_about                      47038
beds                            34920
price                           34908
estimated_revenue_l365d         34908
bathrooms                       34846
host_response_rate              31707
host_response_time              31707
host_acceptance_rate            27760
review_scores_location          24166
review_scores_value             24166
review_scores_checkin           24165
review_scores_communication     24142
review_scores_accuracy          24137
review_scores_cleanliness       24131
last_review                     24122
review_scores_rating            24122
first_review                    24122
reviews_per_month               24122
host_location                   23771
bedrooms                        12775
has_availabi

In [5]:
# Check Duplicate Records
print("Duplicate Rows:", df.duplicated().sum())

Duplicate Rows: 0


In [6]:
# Check Data Types
df.dtypes

id                                                int64
listing_url                                      object
scrape_id                                         int64
last_scraped                                     object
source                                           object
                                                 ...   
calculated_host_listings_count                    int64
calculated_host_listings_count_entire_homes       int64
calculated_host_listings_count_private_rooms      int64
calculated_host_listings_count_shared_rooms       int64
reviews_per_month                               float64
Length: 79, dtype: object

In [8]:
# Handle Missing Values

# Numerical Columns

numerical_columns = [
    "price",
    "bedrooms",
    "beds",
    "bathrooms",
    "reviews_per_month",
    "review_scores_rating"
]

for col in numerical_columns:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Categorical Columns

categorical_columns = [
    "host_is_superhost",
    "host_response_time",
    "host_location"
]

for col in categorical_columns:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

In [9]:
# Remove Unnecessary Columns
columns_to_drop = [
    "license",
    "calendar_updated",
    "neighbourhood_group_cleansed"
]

df.drop(columns=columns_to_drop, inplace=True)

In [11]:
# Convert Data Types

# Dates
date_columns = [
    "host_since",
    "last_scraped",
    "calendar_last_scraped",
    "first_review",
    "last_review"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

# Boolean Columns
df["instant_bookable"] = df["instant_bookable"].map({
    "t": True,
    "f": False
})

df["host_is_superhost"] = df["host_is_superhost"].map({
    "t": True,
    "f": False,
    "Unknown": np.nan
})

In [12]:
# Validate Price
df = df[df["price"] >= 0]

In [13]:
# Check Remaining Missing Values
df.isnull().sum().sort_values(ascending=False).head(20)

neighborhood_overview          55663
neighbourhood                  55662
host_neighbourhood             51021
host_about                     47038
estimated_revenue_l365d        34908
host_response_rate             31707
host_acceptance_rate           27760
review_scores_location         24166
review_scores_value            24166
review_scores_checkin          24165
review_scores_communication    24142
review_scores_accuracy         24137
review_scores_cleanliness      24131
first_review                   24122
last_review                    24122
has_availability                4249
description                     2450
host_is_superhost               1766
bathrooms_text                   153
host_name                         43
dtype: int64

In [14]:
# Save Engineered Dataset
df.to_csv(
    "../data/processed/london_listings_engineered.csv",
    index=False
)

print("Data engineering completed successfully!")

Data engineering completed successfully!
